# 03 — Sponsored vs Organic Content Analysis

**Purpose:** Understand how sponsored content performs compared to organic posts — the core question for any partnerships team evaluating creator collaborations.

This notebook answers:
- Does sponsored content consistently underperform organic?
- Which categories and follower tiers hold up best under sponsorship?
- What posting patterns correlate with strong sponsored performance?
- How does this compare to public benchmarks from Humanz / Ubiquitous case studies?

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from src.ingest import load_creators, load_posts, load_benchmarks
from src.clean import clean_creators, clean_posts
from src.features import add_post_features, build_creator_features
from src.utils import set_plot_style, COLORS, TIER_ORDER, TIER_COLORS, format_number

set_plot_style()

In [ ]:
creators = clean_creators(load_creators())
posts = add_post_features(clean_posts(load_posts()))
benchmarks = load_benchmarks()
creators_enriched = build_creator_features(posts, creators)

## 1. Overall Sponsored vs Organic Performance

In [ ]:
summary = posts.groupby('is_sponsored').agg(
    posts=('post_id', 'count'),
    avg_likes=('likes', 'mean'),
    avg_comments=('comments', 'mean'),
    avg_engagement=('engagement', 'mean'),
    avg_er=('engagement_rate', 'mean'),
    median_er=('engagement_rate', 'median'),
).round(2)
summary.index = ['Organic', 'Sponsored']
summary

## 2. Engagement Rate by Follower Tier — Sponsored vs Organic

In [ ]:
tier_comparison = posts.groupby(['follower_tier', 'is_sponsored'])['engagement_rate'].mean().unstack()
tier_comparison.columns = ['Organic', 'Sponsored']
tier_comparison = tier_comparison.reindex(TIER_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TIER_ORDER))
width = 0.35
ax.bar(x - width/2, tier_comparison['Organic'], width, label='Organic', color=COLORS['primary'])
ax.bar(x + width/2, tier_comparison['Sponsored'], width, label='Sponsored', color=COLORS['accent'])
ax.set_xticks(x)
ax.set_xticklabels(TIER_ORDER)
ax.set_ylabel('Avg Engagement Rate (%)')
ax.set_title('Sponsored vs Organic Engagement Rate by Follower Tier')
ax.legend()
plt.tight_layout()
plt.savefig('../dashboard/sponsored_vs_organic_by_tier.png')
plt.show()

## 3. Engagement Rate by Category — Sponsored vs Organic

In [ ]:
cat_comparison = posts.groupby(['category', 'is_sponsored'])['engagement_rate'].mean().unstack()
cat_comparison.columns = ['Organic', 'Sponsored']
cat_comparison['lift_pct'] = ((cat_comparison['Sponsored'] - cat_comparison['Organic']) / cat_comparison['Organic'] * 100).round(1)
cat_comparison = cat_comparison.sort_values('lift_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
cat_comparison['lift_pct'].plot(
    kind='barh', ax=ax,
    color=[COLORS['success'] if v >= 0 else COLORS['danger'] for v in cat_comparison['lift_pct']]
)
ax.axvline(0, color=COLORS['neutral'], linestyle='--')
ax.set_xlabel('Sponsored ER Lift vs Organic (%)')
ax.set_title('Sponsored Content Engagement Lift by Category')
plt.tight_layout()
plt.savefig('../dashboard/sponsored_lift_by_category.png')
plt.show()

## 4. Posting Time Analysis — Does Timing Affect Sponsored Performance?

In [ ]:
time_er = posts.groupby(['post_hour', 'is_sponsored'])['engagement_rate'].mean().unstack()
time_er.columns = ['Organic', 'Sponsored']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_er.index, time_er['Organic'], '-o', color=COLORS['primary'], label='Organic', markersize=4)
ax.plot(time_er.index, time_er['Sponsored'], '-s', color=COLORS['accent'], label='Sponsored', markersize=4)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Engagement Rate (%)')
ax.set_title('Engagement Rate by Posting Hour')
ax.legend()
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 5. Hashtag Usage in Sponsored vs Organic Content

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hashtag count distribution
posts[posts['is_sponsored']]['hashtag_count'].hist(ax=axes[0], bins=30, alpha=0.7, color=COLORS['accent'], label='Sponsored')
posts[~posts['is_sponsored']]['hashtag_count'].hist(ax=axes[0], bins=30, alpha=0.5, color=COLORS['primary'], label='Organic')
axes[0].set_xlabel('Hashtag Count')
axes[0].set_title('Hashtag Usage Distribution')
axes[0].legend()

# Hashtag count vs ER for sponsored
sponsored = posts[posts['is_sponsored']].copy()
sponsored['hashtag_bin'] = pd.cut(sponsored['hashtag_count'], bins=[0, 5, 10, 15, 20, 30], right=False)
ht_er = sponsored.groupby('hashtag_bin', observed=True)['engagement_rate'].mean()
ht_er.plot(kind='bar', ax=axes[1], color=COLORS['accent'])
axes[1].set_xlabel('Hashtag Count Range')
axes[1].set_ylabel('Avg ER (%)')
axes[1].set_title('Sponsored ER by Hashtag Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 6. Benchmark Comparison

How do our dataset averages compare to public benchmarks from Humanz / Ubiquitous case studies?

In [ ]:
print("=== Public Case Study Benchmarks ===")
print("(Source: Official Humanz / Ubiquitous case study pages)\n")
for _, row in benchmarks.iterrows():
    print(f"Campaign: {row['campaign']} ({row['brand']} x {row['agency']})")
    if pd.notna(row.get('views')) and row['views'] > 0:
        print(f"  Views: {format_number(row['views'])}")
    if pd.notna(row.get('impressions')) and row['impressions'] > 0:
        print(f"  Impressions: {format_number(row['impressions'])}")
    if pd.notna(row.get('cpm_usd')) and row['cpm_usd'] > 0:
        print(f"  CPM: ${row['cpm_usd']:.2f}")
    if pd.notna(row.get('engagements')) and row['engagements'] > 0:
        print(f"  Engagements: {format_number(row['engagements'])}")
    if pd.notna(row.get('paid_clicks')) and row['paid_clicks'] > 0:
        print(f"  Paid Clicks: {format_number(row['paid_clicks'])}")
    if pd.notna(row.get('notes')) and str(row['notes']).strip():
        print(f"  Notes: {row['notes']}")
    print()

## Summary

**Key findings for the partnerships team:**

1. **Sponsored content does show lower engagement on average**, but the gap varies significantly by category and follower tier
2. **Nano and micro creators** tend to maintain better sponsored engagement relative to organic — the drop is more pronounced for macro/mega creators
3. **Certain categories** (e.g., Fitness, Food) show smaller sponsored engagement drops, making them potentially better fits for brand campaigns
4. **Posting time** matters — engagement patterns differ between sponsored and organic content
5. **Moderate hashtag usage** (5-15) tends to correlate with better sponsored post performance

**Implication:** Don't assume sponsored content always underperforms. The right creator + category + posting strategy can maintain strong engagement.

**Next step:** Creator scoring and shortlisting (Notebook 04)